# Qwen3.8-27B Uncensored (HauhauCS Aggressive, MTP) — Colab chat v6: quant picker • context & reasoning • web search • public link

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (preselected; L4 / A100 / H100 / G4 also work and allow a bigger quant and context).
2. **Cell 1** — pick the quant (default **Q4_K_P**, 17.92 GB). On a free T4, Q4_K_P cannot fit; the notebook detects that **before downloading** and automatically switches to the largest smaller quant that does (IQ3_XS normally, Q2_K_P with vision). Then run it: it builds llama.cpp, downloads the model and the tools *in parallel* (~10–25 min the first time; re-running is fast).
3. **Cell 2** — pick the **context window**, **reasoning**, **MTP** and **web search**, then run it. It shows the chat right in the notebook, plus a private link and a **public link**.
4. **Keep this tab open and leave Cell 2 running.** It watches the model server and the link and repairs them automatically.

**Model sizes in this repo (all keep the native MTP head)**

| Quant | Size | Best for |
|---|---|---|
| Q8_K_P | 31.46 GB | A100 80 GB / H100, full context |
| Q6_K_P | 25.92 GB | A100 40 GB+ |
| Q5_K_P | 20.22 GB | L4 24 GB (tight), A100 |
| **Q4_K_P** | **17.92 GB** | **default** — L4 ~128K context, A100 full 262,144 |
| IQ4_XS | 15.71 GB | L4 (smaller than Q4; similar context) |
| Q3_K_P | 13.44 GB | T4 (tight) |
| IQ3_M | 12.79 GB | T4 |
| IQ3_XS | 12.18 GB | T4 — best context (auto fallback) |
| Q2_K_P | 10.68 GB | T4 with vision / maximum context |
| IQ2_M | 10.32 GB | smallest, lowest quality |

**GPU guide (estimates — Cell 2 measures free VRAM and adjusts)**
- **T4 15 GB:** Q4_K_P does **not** fit. Auto fallback picks IQ3_XS without vision, or Q2_K_P with vision. Expect roughly 48K–64K context with built-in MTP; turning MTP off frees a little more.
- **L4 24 GB:** Q4_K_P and IQ4_XS fit roughly 128K; IQ3_M / IQ3_XS can reach the full 262,144.
- **A100 40 GB:** Q4_K_P at the full 262,144 with 8-bit chat memory (tight); use Q3/IQ3 for headroom.
- **A100 80 GB / H100 / G4:** Q8_K_P at full context.

**Reasoning (the model accepts only xhigh / medium / low)**
- **Max** → `xhigh`, no thinking limit (default).
- **High** → `xhigh`, capped at 8,192 thinking tokens.
- **Medium** → `medium`, capped at 2,048.
- **Low** → `low`, capped at 512.
- **Off** → `enable_thinking=false`, plus the model card's non-thinking sampling: **temp 0.7, top_p 0.80, top_k 20, presence 1.5**.
- Sending `high` or `max` directly to `reasoning_effort` makes this model's chat template throw an error. The notebook maps the picker and never sends them.

**MTP (speed)**
- **Built-in MTP** is on by default (`--spec-type draft-mtp`): about **1.6–2.2x** faster than no MTP.
- **FastMTP** (optional, tick in Cell 1) builds the official patched llama.cpp and downloads the 903 MB sidecar: up to roughly **1.9–3x**. If the patch, build or load fails, it automatically falls back to the stock build and built-in MTP.
- Set **MTP → Off** in Cell 2 to save ~0.5–1.5 GB of GPU memory for context.

**Vision**
- Tick **VISION** in Cell 1 to download the 931 MB BF16 projector and attach images. It uses ~1 GB of GPU memory; on a T4 the auto fallback picks a smaller quant.

**Web search**
- **Parallel** is free without a key; if it's busy, DuckDuckGo/Exa are tried automatically. A Parallel key raises limits and adds the deep `research` tool.
- The first search asks permission in the chat — click **Always allow**.
- Long input (1,000+ lines of code or text) is fine: paste it or attach the file.

**Public link**
- **Auto** uses **ngrok** if you paste a free token (dashboard.ngrok.com → Your Authtoken, or a Colab secret `NGROK_AUTHTOKEN`). Its address **never changes** and it's the most reliable.
- Without a token it uses a **Cloudflare** link (no account, new address each run).
- Anyone with the link can use your session. Chats are stored in your browser tab; if the public link changes, it's a new origin, so export chats you care about from the UI settings.

**Good to know**
- `q8_0 / 8-bit chat memory` in the status is the conversation cache, **not** the model file.
- This is an uncensored, "Aggressive" model. You are responsible for how you use its output.


In [ ]:
#@title 1️⃣ Setup — choose the model size, then run (≈10–25 min the first time) { display-mode: "form" }
MODEL_QUANT = "Q4_K_P · 17.92 GB · default (best quality that fits most GPUs)" #@param ["Q4_K_P · 17.92 GB · default (best quality that fits most GPUs)", "Q8_K_P · 31.46 GB · max quality (A100 80 GB / H100)", "Q6_K_P · 25.92 GB · near-lossless (A100 40 GB+)", "Q5_K_P · 20.22 GB · high quality (L4 24 GB, tight)", "IQ4_XS · 15.71 GB · good quality (L4 24 GB)", "Q3_K_P · 13.44 GB · balanced (T4 15 GB, tight)", "IQ3_M · 12.79 GB · smaller (T4 15 GB)", "IQ3_XS · 12.18 GB · best T4 context", "Q2_K_P · 10.68 GB · smallest usable", "IQ2_M · 10.32 GB · lowest quality"]
VISION = False #@param {type:"boolean"}
FASTMTP = False #@param {type:"boolean"}
AUTO_QUANT_FALLBACK = True #@param {type:"boolean"}
USE_DRIVE_CACHE = False #@param {type:"boolean"}
# VISION: also downloads the 931 MB BF16 image projector so you can attach pictures (uses ~1 GB of GPU memory).
# FASTMTP: optional extra speed-up. Builds the official patched llama.cpp and downloads the 903 MB FastMTP sidecar.
#   If the patch/build/load fails, the notebook automatically falls back to the stock build and built-in MTP.
# AUTO_QUANT_FALLBACK: if the picked quant cannot fit this GPU with a 16K context, the largest smaller quant that
#   fits is chosen automatically BEFORE downloading. Strongly recommended: Q4_K_P cannot fit a free T4.
# USE_DRIVE_CACHE: saves the compiled server to Google Drive so the next session skips the ~10–25 min build.

import os, re, sys, time, json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

# ------------------------------------------------------------------ model
# Real sizes from the HauhauCS model card (decimal GB).  Q4_K_P is the default.
QUANT_INFO = {
    "Q8_K_P": 31.46,
    "Q6_K_P": 25.92,
    "Q5_K_P": 20.22,
    "Q4_K_P": 17.92,
    "IQ4_XS": 15.71,
    "Q3_K_P": 13.44,
    "IQ3_M":  12.79,
    "IQ3_XS": 12.18,
    "Q2_K_P": 10.68,
    "IQ2_M":  10.32,
}
QUANT = MODEL_QUANT.split()[0]
if QUANT not in QUANT_INFO:
    raise RuntimeError(f"Unknown quant {QUANT!r}. Pick one from the dropdown.")

REPO_ID   = "HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF"
REPO_REV  = "993a5971fda8f30dd1b7eb2654792ba4415c7460"   # pinned repo revision (reproducible downloads)
FILENAME  = f"Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-{QUANT}.gguf"
MMPROJ    = "mmproj-Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-BF16.gguf"
FASTMTP_FILE = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-FastMTP-32K.gguf"
PATCH_URL = f"https://huggingface.co/{REPO_ID}/resolve/{REPO_REV}/HauhauCS-FastMTP-llama.cpp.patch"

# llama.cpp: stock build for built-in MTP; official pinned commit for the FastMTP patch.
LLAMA_TAG       = "b11925"                                  # tested stock release (has Qwen3.8 + MTP flags)
FASTMTP_COMMIT  = "4df29be4f4c3673f428170fda944a5b19f743bb8"  # exact base required by HauhauCS-FastMTP-llama.cpp.patch
UI_TAG_STOCK    = "b11925"                                  # prebuilt web UI for the stock build
UI_TAG_PATCHED  = "b10532"                                  # closest release after the patch base
MAX_NATIVE_CTX  = 262144

LLAMA_DIR = "/content/llama.cpp"
BIN_PATH  = f"{LLAMA_DIR}/build/bin/llama-server"
STAMP     = f"{LLAMA_DIR}/build/qwen38_build_stamp.txt"
MODEL_DIR = "/content/models"
CF_PATH   = "/content/cloudflared"
CONFIG    = "/content/qwen38_config.json"
PATCH_PATH = "/content/HauhauCS-FastMTP-llama.cpp.patch"

os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------------ GPU / RAM
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                   "--format=csv,noheader"], text=True).strip().splitlines()[0]
    gpu_name, cap, vram = [x.strip() for x in gpu.split(",")]
    ARCH = cap.replace(".", "")
    VRAM_MIB = int(round(float(re.search(r"[\d.]+", vram).group()) *
                         (1024 if "gib" in vram.lower() else 1)))
    print(f"✅ GPU: {gpu_name} (sm_{ARCH}, {VRAM_MIB/1024:.1f} GB)")
except Exception:
    raise RuntimeError("❌ No GPU found. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

try:
    RAM_GIB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
except Exception:
    RAM_GIB = 16.0
CACHE_RAM_MIB = 1024 if RAM_GIB < 20 else 2048
BUILD_JOBS = max(1, min(os.cpu_count() or 2, 8))
if RAM_GIB < 16:
    BUILD_JOBS = min(BUILD_JOBS, 4)

# ------------------------------------------------------------------ fit check (before downloading)
VISION_MIB       = 1024          # 931 MB projector + its compute buffers
FASTMTP_MIB      = 1536          # 903 MB sidecar + its own KV/context
MTP_MIB          = 512           # embedded MTP context on the target weights
BASE_OVERHEAD_MIB = 650
MIN_CTX          = 16384
MIN_FREE_MIB     = 400

def kv_mib(ctx):
    """Conservative 8-bit KV estimate: Qwen3.8 has 16 attention layers out of 64."""
    return ctx / 1024 * 34

def quant_fits(q, vision=None, fastmtp=None):
    """Conservative check used to choose an automatic fallback (7% size margin + 5% VRAM headroom)."""
    vision  = VISION  if vision  is None else vision
    fastmtp = FASTMTP if fastmtp is None else fastmtp
    need = (QUANT_INFO[q] * 1024 + BASE_OVERHEAD_MIB +
            (VISION_MIB if vision else 0) + (FASTMTP_MIB if fastmtp else 0) +
            kv_mib(MIN_CTX) + MIN_FREE_MIB)
    return need <= VRAM_MIB * 0.95

def quant_fits_tight(q, vision=None, fastmtp=None):
    """Physical fit using the exact decimal-GB file sizes, including the default built-in MTP context.

    This is used to honor a quant the user explicitly picked; Cell 2 still lowers the
    context if the real allocation ends up too tight.
    """
    vision  = VISION  if vision  is None else vision
    fastmtp = FASTMTP if fastmtp is None else fastmtp
    need = (QUANT_INFO[q] * 1e9 / 2**20 + BASE_OVERHEAD_MIB +
            (VISION_MIB if vision else 0) +
            (FASTMTP_MIB if fastmtp else MTP_MIB) +
            kv_mib(MIN_CTX) + MIN_FREE_MIB)
    return need <= VRAM_MIB

# If FastMTP cannot fit even at the physical limit with the picked quant, drop the
# optional sidecar before downgrading model quality.
if FASTMTP and not quant_fits_tight(QUANT, fastmtp=True):
    FASTMTP = False
    print("ℹ️ Not enough GPU memory for the optional FastMTP sidecar with this quant — using built-in MTP instead.")

tight_fit = quant_fits_tight(QUANT)
if not quant_fits(QUANT):
    if tight_fit:
        print("ℹ️ Tight fit: this quant physically fits a 16K context, but with little spare VRAM. "
              "Cell 2 will lower the context if needed.")
    else:
        if not AUTO_QUANT_FALLBACK:
            raise RuntimeError(
                f"❌ {QUANT} ({QUANT_INFO[QUANT]:.2f} GB) cannot fit this {VRAM_MIB/1024:.1f} GB GPU with a 16K context.\n"
                f"   Pick a smaller quant in Cell 1 (IQ3_XS or Q2_K_P for a T4), or tick AUTO_QUANT_FALLBACK.")
        smaller = sorted((q for q in QUANT_INFO
                          if QUANT_INFO[q] < QUANT_INFO[QUANT] and quant_fits(q)),
                         key=lambda q: QUANT_INFO[q], reverse=True)
        if not smaller:
            raise RuntimeError(
                f"❌ Even IQ2_M does not fit this {VRAM_MIB/1024:.1f} GB GPU with VISION={VISION}, FASTMTP={FASTMTP}.\n"
                "   Turn those options off or use a larger GPU.")
        old = QUANT
        QUANT = smaller[0]
        FILENAME = f"Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-{QUANT}.gguf"
        print(f"🔄 Auto fallback: {old} → {QUANT}.  {old} needs ~{QUANT_INFO[old]*1.07 + 1.1:.1f} GB "
              f"with a 16K context; this GPU has {VRAM_MIB/1024:.1f} GB.")

print(f"🎛  Quant: {QUANT} ({QUANT_INFO[QUANT]:.2f} GB)  |  vision: {'ON' if VISION else 'off'}"
      f"  |  FastMTP: {'ON (patched build)' if FASTMTP else 'off (built-in MTP)'}")

# ------------------------------------------------------------------ disk
need_disk = QUANT_INFO[QUANT] + (0.95 if VISION else 0) + (0.95 if FASTMTP else 0) + 8
free_gb = shutil.disk_usage("/content").free / 1e9
if free_gb < need_disk:
    raise RuntimeError(f"❌ Only {free_gb:.0f} GB free on /content; this setup needs ~{need_disk:.0f} GB. "
                       "Delete old files (Runtime → Manage sessions) or pick a smaller quant.")
print(f"💾 Disk: {free_gb:.0f} GB free (need ~{need_disk:.0f} GB)")

# ------------------------------------------------------------------ optional Drive cache
CACHE_DIR  = "/content/drive/MyDrive/qwen38_cache"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(variant):
    ref = FASTMTP_COMMIT if variant == "fastmtp" else LLAMA_TAG
    return f"{CACHE_DIR}/llama-server-{variant}-{ref[:9]}-sm{ARCH}"

# ------------------------------------------------------------------ helpers
def run(cmd, log, env=None, retries=1):
    for attempt in range(retries):
        with open(log, "a") as f:
            f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        if p.returncode == 0:
            return
        if attempt + 1 < retries:
            time.sleep(5)
    tail = open(log, errors="replace").read()[-4000:]
    raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{tail}")

def binary_ok(path):
    try:
        out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60)
        return out.returncode == 0
    except Exception:
        return False

# ------------------------------------------------------------------ build
def build_server(variant):
    ref     = FASTMTP_COMMIT if variant == "fastmtp" else LLAMA_TAG
    ui_tag  = UI_TAG_PATCHED if variant == "fastmtp" else UI_TAG_STOCK
    want    = f"{variant}:{ref[:9]}:sm{ARCH}"
    if os.path.exists(BIN_PATH) and os.path.exists(STAMP) and open(STAMP).read().strip() == want and binary_ok(BIN_PATH):
        return "already built"
    cached = cache_path(variant)
    if USE_DRIVE_CACHE and os.path.exists(cached):
        try:
            os.makedirs(os.path.dirname(BIN_PATH), exist_ok=True)
            shutil.copy(cached, BIN_PATH); os.chmod(BIN_PATH, 0o755)
            if binary_ok(BIN_PATH):
                open(STAMP, "w").write(want)
                return "restored from Drive cache"
        except Exception as e:
            print(f"   (Drive cache restore failed: {e}; building instead)")
    log = "/content/build.log"
    open(log, "w").close()
    shutil.rmtree(LLAMA_DIR, ignore_errors=True)          # always start from a clean pinned source

    if variant == "fastmtp":
        os.makedirs(LLAMA_DIR, exist_ok=True)
        run(["git", "init", "-q", LLAMA_DIR], log)
        run(["git", "-C", LLAMA_DIR, "remote", "add", "origin", "https://github.com/ggml-org/llama.cpp"], log)
        run(["git", "-C", LLAMA_DIR, "fetch", "--depth", "1", "origin", FASTMTP_COMMIT], log, retries=3)
        run(["git", "-C", LLAMA_DIR, "checkout", "-q", "--detach", "FETCH_HEAD"], log)
    else:
        # A half-finished clone cannot be resumed; clean it before each attempt.
        for attempt in range(3):
            shutil.rmtree(LLAMA_DIR, ignore_errors=True)
            try:
                run(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG,
                     "https://github.com/ggml-org/llama.cpp", LLAMA_DIR], log)
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(5)

    # Web UI: prebuilt tarball for the matching release (no Node.js build needed).
    dist = f"{LLAMA_DIR}/tools/ui/dist"
    os.makedirs(dist, exist_ok=True)
    try:
        run(["wget", "-q", "--tries=3", "-O", "/content/llama-ui.tar.gz",
             f"https://github.com/ggml-org/llama.cpp/releases/download/{ui_tag}/llama-{ui_tag}-ui.tar.gz"], log)
        run(["tar", "-xzf", "/content/llama-ui.tar.gz", "-C", dist, "--strip-components=1"], log)
    except Exception:
        shutil.rmtree(dist, ignore_errors=True)           # cmake then downloads the same UI from Hugging Face

    if variant == "fastmtp":
        run(["wget", "-q", "--tries=3", "-O", PATCH_PATH, PATCH_URL], log, retries=3)
        run(["git", "-C", LLAMA_DIR, "apply", "--check", PATCH_PATH], log)
        run(["git", "-C", LLAMA_DIR, "apply", PATCH_PATH], log)

    env = dict(os.environ, HF_UI_VERSION=ui_tag)
    run(["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build",
         "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={ARCH}",
         "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_OPENSSL=OFF",
         "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
         "-DCMAKE_BUILD_TYPE=Release"], log, env=env)
    run(["cmake", "--build", f"{LLAMA_DIR}/build", "--config", "Release",
         "--target", "llama-server", "-j", str(BUILD_JOBS)], log, env=env)
    if not binary_ok(BIN_PATH):
        raise RuntimeError("The server was built but does not start. See /content/build.log")
    open(STAMP, "w").write(want)
    if USE_DRIVE_CACHE:
        try:
            os.makedirs(CACHE_DIR, exist_ok=True)
            shutil.copy(BIN_PATH, cached)
        except Exception as e:
            print(f"   (Drive cache write failed: {e})")
    return "built from source"

build_notes = []

def build_with_fallback():
    if not FASTMTP:
        return build_server("stock"), "stock"
    try:
        return build_server("fastmtp"), "fastmtp"
    except Exception as e:
        build_notes.append(f"FastMTP build failed: {str(e)[:700]}")
        return build_server("stock"), "stock"

# ------------------------------------------------------------------ tools
def get_cloudflared():
    if os.path.exists(CF_PATH) and os.path.getsize(CF_PATH) > 1_000_000 and binary_ok(CF_PATH):
        return "already there"
    run(["wget", "-q", "--tries=3", "-O", CF_PATH,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        "/content/cf.log", retries=3)
    os.chmod(CF_PATH, 0o755)
    if os.path.getsize(CF_PATH) < 1_000_000 or not binary_ok(CF_PATH):
        os.remove(CF_PATH)
        raise RuntimeError("cloudflared download looks broken; run this cell again.")
    return "downloaded"

def get_web_deps():
    # DuckDuckGo search + page-to-text, used by the DuckDuckGo option and as automatic fallback.
    p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ddgs", "trafilatura", "pyngrok"],
                       capture_output=True, text=True)
    return "installed" if p.returncode == 0 else "skipped (pip failed; Parallel/Exa still work)"

# ------------------------------------------------------------------ download
def download_model():
    try:
        import huggingface_hub  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download, list_repo_files
    token = None
    try:  # optional Colab secret HF_TOKEN (only needed if the repo ever becomes gated)
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

    fname = FILENAME
    try:
        files = list_repo_files(REPO_ID, revision=REPO_REV, token=token)
        if fname not in files:
            cands = sorted(f for f in files
                           if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                           and "fastmtp" not in f.lower()
                           and f.lower().endswith(f"-{QUANT.lower()}.gguf"))
            if not cands:
                raise RuntimeError(f"No {QUANT} file in {REPO_ID}. Files:\n" + "\n".join(files))
            fname = cands[0]
    except RuntimeError:
        raise
    except Exception as e:
        print(f"   (could not list repo files: {e}; trying the standard file name)")

    # Free disk: remove other quants from earlier runs (keeps the projector and FastMTP sidecar).
    for old in os.listdir(MODEL_DIR):
        p = os.path.join(MODEL_DIR, old)
        if (old.endswith(".gguf") and old != os.path.basename(fname)
                and not old.startswith("mmproj") and "FastMTP" not in old):
            try:
                os.remove(p)
            except OSError:
                pass

    def fetch(name):
        last = None
        for attempt in range(4):
            try:
                return hf_hub_download(REPO_ID, name, revision=REPO_REV, local_dir=MODEL_DIR, token=token)
            except Exception as e:
                last = e
                time.sleep(10 * (attempt + 1))
        raise RuntimeError(f"Download of {name} failed after 4 tries: {last}")

    model = fetch(fname)
    mmproj = fetch(MMPROJ) if VISION else ""
    fastmtp = ""
    if FASTMTP:
        try:
            fastmtp = fetch(FASTMTP_FILE)
        except Exception as e:
            print(f"   ⚠️ FastMTP sidecar download failed ({str(e)[:200]}); built-in MTP will be used.")
    return model, mmproj, fastmtp

# ------------------------------------------------------------------ run everything
t0 = time.time()
print(f"⏳ Building llama.cpp and downloading {FILENAME} at the same time...")
with ThreadPoolExecutor(max_workers=4) as ex:
    jobs = {"build": ex.submit(build_with_fallback), "download": ex.submit(download_model),
            "tunnel tool": ex.submit(get_cloudflared), "web tools": ex.submit(get_web_deps)}
    last = -1
    while not all(j.done() for j in jobs.values()):
        time.sleep(5)
        mins = int((time.time() - t0) // 60)
        if mins != last:
            last = mins
            print(f"   {mins:>2} min | " + " | ".join(f"{k}: {'done' if j.done() else '…'}" for k, j in jobs.items()))
    errors = []
    for k, j in jobs.items():
        if j.exception() is not None and k != "web tools":
            errors.append(f"[{k}] {j.exception()}")
    if errors:
        raise RuntimeError("Setup failed — just run this cell again (finished parts are reused).\n\n" + "\n\n".join(errors))
    build_status, build_variant = jobs["build"].result()
    model_path, mmproj_path, fastmtp_path = jobs["download"].result()
    cf_status = jobs["tunnel tool"].result()
    web_status = jobs["web tools"].result() if jobs["web tools"].exception() is None else "skipped"

if build_variant != "fastmtp":
    fastmtp_path = ""

json.dump({"model": model_path, "mmproj": mmproj_path, "fastmtp": fastmtp_path,
           "quant": QUANT, "bin": BIN_PATH, "cloudflared": CF_PATH,
           "tag": FASTMTP_COMMIT if build_variant == "fastmtp" else LLAMA_TAG,
           "build": build_variant, "max_ctx": MAX_NATIVE_CTX,
           "vram_mib": VRAM_MIB, "ram_gib": round(RAM_GIB, 1),
           "cache_ram_mib": CACHE_RAM_MIB}, open(CONFIG, "w"))

print(f"\n✅ Server:      {build_status} (llama.cpp {LLAMA_TAG if build_variant == 'stock' else FASTMTP_COMMIT[:9]}"
      f"{' + FastMTP patch' if build_variant == 'fastmtp' else ''})")
print(f"✅ Model:       {os.path.basename(model_path)} ({os.path.getsize(model_path)/1e9:.2f} GB)")
if mmproj_path:
    print(f"✅ Vision:      {os.path.basename(mmproj_path)}")
print(f"✅ Tunnel tool: {cf_status}")
print(f"✅ Web tools:   {web_status}")
if FASTMTP and build_variant == "fastmtp" and fastmtp_path:
    print(f"✅ FastMTP:     {os.path.basename(fastmtp_path)} (patched build)")
elif FASTMTP:
    print("ℹ️ FastMTP was requested but is not active — Cell 2 will use built-in MTP.")
for note in build_notes:
    print(f"⚠️ {note}")
print(f"⏱  {int(time.time() - t0)//60} min {int(time.time() - t0)%60} s.  👉 Now run Cell 2.")


In [ ]:
#@title 2️⃣ Launch — choose context, reasoning, MTP and web search, then run (keep this cell running!) { display-mode: "form" }
CONTEXT_WINDOW = "262144 · max (model limit, default)" #@param ["262144 · max (model limit, default)", "196608", "131072", "98304", "65536", "49152", "32768", "16384", "8192", "4096"]
REASONING = "Max · xhigh, unlimited thinking (default)" #@param ["Max · xhigh, unlimited thinking (default)", "High · xhigh, capped at 8192 thinking tokens", "Medium · medium, capped at 2048 thinking tokens", "Low · low, capped at 512 thinking tokens", "Off · answers directly, no thinking"]
MTP = "On · FastMTP if built, else built-in (default)" #@param ["On · FastMTP if built, else built-in (default)", "Built-in MTP only · safest", "Off · save GPU memory"]
WEB_SEARCH = "Parallel · free, no key needed (default)" #@param ["Parallel · free, no key needed (default)", "Exa · free tier is rate-limited, best with a key", "DuckDuckGo · free, no key needed", "Off · no web tools"]
AUTO_FALLBACK = True #@param {type:"boolean"}
PARALLEL_API_KEY = "" #@param {type:"string"}
EXA_API_KEY = "" #@param {type:"string"}
PUBLIC_LINK = "Auto · ngrok if a token is set, else Cloudflare (default)" #@param ["Auto · ngrok if a token is set, else Cloudflare (default)", "ngrok · same link every time, free account", "Cloudflare · no account, new link each run", "Off · private Colab link only"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}
SHOW_INLINE = True #@param {type:"boolean"}
# CONTEXT_WINDOW: Cell 2 estimates what fits in the GPU and automatically uses the next smaller size if needed.
# REASONING: the model accepts only xhigh / medium / low. Max→xhigh, High→xhigh (capped), Medium→medium, Low→low,
#   Off→enable_thinking=false and the model card's non-thinking sampling (temp 0.7, top_p 0.80, presence 1.5).
# MTP: built-in MTP is ~1.6–2.2x faster than no MTP. FastMTP (built in Cell 1) adds more; it falls back automatically.
# AUTO_FALLBACK: if the chosen search service is down or rate-limited, the other free ones are tried automatically.
# API keys are optional (you can also add them as Colab secrets named PARALLEL_API_KEY / EXA_API_KEY).
# A Parallel key raises limits and adds a deep "research" tool.
# PUBLIC_LINK: anyone with the link can use your session. ngrok = most reliable: free token from
#   https://dashboard.ngrok.com/get-started/your-authtoken (or Colab secret NGROK_AUTHTOKEN); your link never changes.
# SHOW_INLINE: also shows the chat right here in the notebook (private, always works).

# ---------------- advanced (rarely needed) ----------------
KV_CACHE_TYPE = "q8_0"          # "q8_0" halves context memory vs "f16" with almost no quality loss
PRESENCE_PENALTY_THINKING    = 0.0   # raise to 1.0 if thinking ever loops/repeats
PRESENCE_PENALTY_NONTHINKING = 1.5   # model card value for non-thinking mode
MAX_SERVER_RESTARTS = 5
MTP_DEPTH_FASTMTP   = 3         # draft depth: model card uses 3 for FastMTP
MTP_DEPTH_EMBEDDED  = 2         # model card uses 2 for built-in MTP
# -----------------------------------------------------------

import os, re, sys, json, time, datetime, subprocess
import requests
from IPython.display import display, HTML

import socket
def free_port(start):
    """First port from `start` that nothing else (e.g. Colab's own services) is using."""
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sk:
            try:
                sk.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"No free port found near {start}")

LOG, TLOG = "/content/llama-server.log", "/content/tunnel.log"
CONFIG = "/content/qwen38_config.json"
WEB_SCRIPT, MCP_CONFIG = "/content/qwen38_web_mcp.py", "/content/qwen38_mcp.json"
CTX_STEPS = [262144, 196608, 131072, 98304, 65536, 49152, 32768, 24576, 16384, 12288, 8192, 6144, 4096]
MIN_FREE_MIB = 400            # GPU memory that must stay free after loading, so long chats don't crash later
LOCAL = requests.Session()
LOCAL.trust_env = False       # talk to the local server/tunnel directly, never through a proxy

if not os.path.exists(CONFIG):
    raise RuntimeError("Run Cell 1 first.")
cfg = json.load(open(CONFIG))
BIN         = cfg["bin"]
MODEL_PATH  = cfg["model"]
MMPROJ_PATH = cfg.get("mmproj", "")
FASTMTP_PATH = cfg.get("fastmtp", "")
CF          = cfg["cloudflared"]
QUANT       = cfg["quant"]
MAX_NATIVE_CTX = int(cfg.get("max_ctx", 262144))
CACHE_RAM_MIB  = int(cfg.get("cache_ram_mib", 2048))
for p in (BIN, MODEL_PATH):
    if not os.path.exists(p):
        raise RuntimeError(f"Missing {p}. Run Cell 1 again.")
if MMPROJ_PATH and not os.path.exists(MMPROJ_PATH):
    print("⚠️ Vision projector is missing; continuing without vision.")
    MMPROJ_PATH = ""
if FASTMTP_PATH and not os.path.exists(FASTMTP_PATH):
    print("⚠️ FastMTP sidecar is missing; using built-in MTP.")
    FASTMTP_PATH = ""
LINK_MODE = PUBLIC_LINK.split()[0].lower()            # auto | ngrok | cloudflare | off

# ---------------- parse the form ----------------
TARGET_CONTEXT = max(4096, min(int(re.search(r"\d+", CONTEXT_WINDOW.strip()).group()), MAX_NATIVE_CTX))

_r = REASONING.split()[0].lower()
if _r == "off":
    THINKING, EFFORT, REASONING_BUDGET = False, None, 0
elif _r == "max":
    THINKING, EFFORT, REASONING_BUDGET = True, "xhigh", -1
elif _r == "high":
    THINKING, EFFORT, REASONING_BUDGET = True, "xhigh", 8192
elif _r == "medium":
    THINKING, EFFORT, REASONING_BUDGET = True, "medium", 2048
elif _r == "low":
    THINKING, EFFORT, REASONING_BUDGET = True, "low", 512
else:
    raise RuntimeError(f"Unknown reasoning option {REASONING!r}")
if EFFORT not in (None, "xhigh", "medium", "low"):
    raise RuntimeError(f"Unsupported reasoning effort {EFFORT!r}; the model accepts only xhigh, medium and low.")

_mtp = MTP.split()[0].lower()
if _mtp == "off":
    MTP_MODE = "off"
elif _mtp == "built-in":
    MTP_MODE = "embedded"
else:
    MTP_MODE = "fastmtp" if FASTMTP_PATH else "embedded"

WEB_BACKEND = {"parallel": "parallel", "exa": "exa", "duckduckgo": "duckduckgo"}.get(WEB_SEARCH.split()[0].lower(), "off")

def secret(name, typed):
    if typed.strip():
        return typed.strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""
NGROK_TOKEN = secret("NGROK_AUTHTOKEN", NGROK_AUTHTOKEN) if LINK_MODE in ("auto", "ngrok") else ""
if LINK_MODE == "auto":
    LINK_MODE = "ngrok" if NGROK_TOKEN else "cloudflare"
if LINK_MODE == "ngrok" and not NGROK_TOKEN:
    print("⚠️ ngrok needs a free token (NGROK_AUTHTOKEN). Using Cloudflare instead.")
    LINK_MODE = "cloudflare"
if LINK_MODE == "cloudflare" and not os.path.exists(CF):
    raise RuntimeError("cloudflared is missing. Run Cell 1 again (or set PUBLIC_LINK to Off).")
PKEY = secret("PARALLEL_API_KEY", PARALLEL_API_KEY) if WEB_BACKEND != "off" else ""
EKEY = secret("EXA_API_KEY", EXA_API_KEY) if WEB_BACKEND != "off" else ""

# ---------------- web tools (stdio MCP server run by llama-server) ----------------
WEB_MCP_SOURCE = r'''#!/usr/bin/env python3
"""Qwen3.8 web tools: a tiny stdio MCP server for llama-server (--mcp-servers-config).

Gives the model two tools, web_search and web_fetch, backed by:
  parallel   - Parallel MCP (https://mcp.parallel.ai/mcp), free without a key
  exa        - Exa MCP (https://mcp.exa.ai/mcp), free tier is rate limited, key optional
  duckduckgo - DuckDuckGo search (ddgs) + direct page download, no key
If the chosen backend fails (rate limit, network), it falls back to the next one
automatically, so a search almost never comes back empty.

Settings come from environment variables set by the notebook:
  WEB_BACKEND        parallel | exa | duckduckgo        (default parallel)
  PARALLEL_API_KEY   optional, higher limits + adds web_research
  EXA_API_KEY        optional
  WEB_FALLBACK       1 = try other backends on failure  (default 1)
  WEB_MAX_CHARS      cap on characters returned per call (default 24000)
Only JSON-RPC goes to stdout; diagnostics go to stderr.
"""
import json, os, re, sys, time, html, urllib.request, urllib.error

BACKEND = os.environ.get("WEB_BACKEND", "parallel").strip().lower() or "parallel"
PARALLEL_KEY = os.environ.get("PARALLEL_API_KEY", "").strip()
EXA_KEY = os.environ.get("EXA_API_KEY", "").strip()
FALLBACK = os.environ.get("WEB_FALLBACK", "1") != "0"
MAX_CHARS = int(os.environ.get("WEB_MAX_CHARS", "24000") or 24000)
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"


def log(*a):
    print("[qwen38-web]", *a, file=sys.stderr, flush=True)


# ----------------------------------------------------------------- remote MCP client
class RemoteMCP:
    """Minimal Streamable-HTTP MCP client (JSON or SSE replies)."""

    def __init__(self, url, headers=None, timeout=90):
        self.url, self.timeout, self.sid = url, timeout, None
        self.headers = {"Content-Type": "application/json",
                        "Accept": "application/json, text/event-stream",
                        "User-Agent": "qwen38-colab/6"}
        self.headers.update(headers or {})
        self._id = 0

    def _post(self, payload, want_reply=True):
        h = dict(self.headers)
        if self.sid:
            h["Mcp-Session-Id"] = self.sid
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode("utf-8"),
                                     headers=h, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                sid = r.headers.get("Mcp-Session-Id")
                if sid:
                    self.sid = sid
                ctype = (r.headers.get("Content-Type") or "").lower()
                body = r.read().decode("utf-8", errors="replace")   # always UTF-8 (fixes mojibake)
        except urllib.error.HTTPError as e:
            detail = ""
            try:
                detail = e.read().decode("utf-8", errors="replace")[:500]
            except Exception:
                pass
            raise RuntimeError(f"HTTP {e.code} from {self.url}: {detail}")
        if not want_reply:
            return None
        msgs = []
        if "event-stream" in ctype:
            data_lines = []
            for line in body.splitlines() + [""]:          # SSE: blank line ends an event
                if line.startswith("data:"):
                    data_lines.append(line[5:].lstrip(" "))
                elif line.strip() == "":
                    if data_lines:
                        try:
                            msgs.append(json.loads("\n".join(data_lines)))
                        except ValueError:
                            pass
                        data_lines = []
        elif body.strip():
            j = json.loads(body)
            msgs = j if isinstance(j, list) else [j]
        for m in msgs:
            if isinstance(m, dict) and m.get("id") == payload.get("id"):
                return m
        for m in msgs:                                      # some servers reply with id null on errors
            if isinstance(m, dict) and "error" in m:
                return m
        raise RuntimeError("No reply from MCP server")

    def _rpc(self, method, params=None):
        self._id += 1
        reply = self._post({"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}})
        if "error" in reply:
            err = reply["error"]
            raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
        return reply.get("result", {})

    def call(self, tool, args):
        self._rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                 "clientInfo": {"name": "qwen38-colab", "version": "6"}})
        try:
            self._post({"jsonrpc": "2.0", "method": "notifications/initialized"}, want_reply=False)
        except Exception:
            pass
        res = self._rpc("tools/call", {"name": tool, "arguments": args})
        text = result_text(res)
        if res.get("isError"):
            raise RuntimeError(text or "tool error")
        return text


def result_text(res):
    parts = []
    for c in res.get("content") or []:
        if isinstance(c, dict) and c.get("type") == "text" and c.get("text"):
            parts.append(c["text"])
    if not parts and res.get("structuredContent") is not None:
        parts.append(json.dumps(res["structuredContent"], ensure_ascii=False, indent=1))
    return "\n".join(parts).strip()


# ----------------------------------------------------------------- backends
def _queries(a):
    q = a.get("query") or ""
    qs = a.get("search_queries") or []
    if isinstance(qs, str):
        qs = [qs]
    qs = [str(x).strip() for x in qs if str(x).strip()]
    if q.strip() and q.strip() not in qs:
        qs.insert(0, q.strip())
    return qs[:5]


def _urls(a):
    u = a.get("urls") or a.get("url") or []
    if isinstance(u, str):
        u = re.split(r"[\s,]+", u)
    out = []
    for x in u:
        x = str(x).strip().strip("<>\"'")
        if x and not re.match(r"^https?://", x, re.I):
            x = "https://" + x
        if x and x not in out:
            out.append(x)
    return out[:10]


def parallel_search(a):
    qs = _queries(a)
    objective = (a.get("objective") or "").strip() or (qs[0] if qs else "")
    if not objective:
        raise RuntimeError("web_search needs a query")
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr).call(
        "web_search", {"objective": objective[:4000], "search_queries": qs or [objective[:200]]})


def parallel_fetch(a):
    args = {"urls": _urls(a)}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:200]
    if a.get("full_content"):
        args["full_content"] = True
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr).call("web_fetch", args)


def parallel_research(a):
    if not PARALLEL_KEY:
        raise RuntimeError("deep research needs a Parallel API key")
    hdr = {"Authorization": f"Bearer {PARALLEL_KEY}"}
    args = {"input": str(a.get("question") or a.get("input") or a.get("query") or "")[:20000]}
    return RemoteMCP("https://mcp.parallel.ai/mcp", hdr, timeout=280).call("web_research", args)


def _exa():
    hdr = {"Authorization": f"Bearer {EXA_KEY}"} if EXA_KEY else {}
    return RemoteMCP("https://mcp.exa.ai/mcp", hdr)


def exa_search(a):
    qs = _queries(a)
    # Exa's schema wants an objective as well as the query (it also works without one,
    # but sending it improves ranking and avoids a schema-validation error on stricter days).
    objective = str(a.get("objective") or "Find the most relevant, authoritative pages for this query.")[:4000]
    args = {"query": " ".join(qs[:1]) or objective, "objective": objective, "numResults": 8}
    return _exa().call("web_search_exa", args)


def exa_fetch(a):
    return _exa().call("web_fetch_exa", {"urls": _urls(a),
                                         "maxCharacters": 60000 if a.get("full_content") else 8000})


def ddg_search(a):
    try:
        from ddgs import DDGS
    except ImportError:
        raise RuntimeError("ddgs is not installed (pip install ddgs)")
    out, seen = [], set()
    queries = _queries(a)[:3]
    if not queries and a.get("objective"):
        queries = [str(a["objective"]).strip()]
    if not queries:
        raise RuntimeError("web_search needs a query")
    for q in queries:
        for r in DDGS().text(q, max_results=8) or []:
            url = r.get("href") or r.get("url")
            if not url or url in seen:
                continue
            seen.add(url)
            out.append(f"## {r.get('title','')}\nURL: {url}\n{r.get('body','')}")
    if not out:
        raise RuntimeError("DuckDuckGo returned no results")
    return "\n\n".join(out)


def _download(url):
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html,*/*;q=0.8"})
    with urllib.request.urlopen(req, timeout=30) as r:
        raw = r.read(6_000_000)
        cs = r.headers.get_content_charset() or "utf-8"
        ctype = (r.headers.get("Content-Type") or "").lower()
    txt = raw.decode(cs, errors="replace")
    if "html" not in ctype and "<html" not in txt[:2000].lower():
        return txt
    try:
        import trafilatura
        ext = trafilatura.extract(txt, url=url, include_links=False, include_tables=True,
                                  output_format="markdown")
        if ext and len(ext) > 200:
            return ext
    except Exception:
        pass
    txt = re.sub(r"(?is)<(script|style|noscript|svg|head)[^>]*>.*?</\1>", " ", txt)
    txt = re.sub(r"(?s)<[^>]+>", " ", txt)
    return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n\s*\n+", "\n\n", html.unescape(txt))).strip()


def direct_fetch(a):
    urls, parts, ok = _urls(a), [], 0
    if not urls:
        raise RuntimeError("no valid URL given")
    per = max(3000, MAX_CHARS // len(urls))
    for u in urls:
        try:
            parts.append(f"# {u}\n{_download(u)[:per]}")
            ok += 1
        except Exception as e:
            parts.append(f"# {u}\n(could not load: {e})")
    if not ok:
        raise RuntimeError("could not load any URL - " + " | ".join(parts)[:800])
    return "\n\n".join(parts)


SEARCH = {"parallel": parallel_search, "exa": exa_search, "duckduckgo": ddg_search}
FETCH = {"parallel": parallel_fetch, "exa": exa_fetch, "duckduckgo": direct_fetch}


def run_with_fallback(table, args):
    order = [BACKEND] + ([b for b in ("parallel", "duckduckgo", "exa") if b != BACKEND] if FALLBACK else [])
    errors = []
    for b in order:
        if b not in table:
            continue
        msg = "unknown error"
        for attempt in range(2):                     # one quick retry per backend
            try:
                text = table[b](args)
                if text and text.strip():
                    note = "" if b == BACKEND else f"[results from {b}; {BACKEND} was unavailable]\n"
                    return note + text
                raise RuntimeError("empty result")
            except Exception as e:
                msg = str(e)
                log(f"{b} attempt {attempt + 1} failed: {msg[:300]}")
                if "rate limit" in msg.lower() or "HTTP 4" in msg:
                    break                            # don't hammer a rate-limited backend
                time.sleep(1.5)
        errors.append(f"{b}: {msg[:200]}")
    raise RuntimeError("all web backends failed - " + " | ".join(errors))


# ----------------------------------------------------------------- MCP server side
TOOLS = [
    {"name": "web_search",
     "description": ("Search the web for current information. Returns titles, URLs and text excerpts. "
                     "Use for anything recent, factual, or that you are unsure about."),
     "inputSchema": {"type": "object", "properties": {
         "query": {"type": "string", "description": "Short keyword search query, 3-8 words."},
         "objective": {"type": "string", "description": "Optional: one sentence on what you want to find."}},
         "required": ["query"]}},
    {"name": "web_fetch",
     "description": "Read one or more web pages (e.g. URLs from web_search) and return their main text.",
     "inputSchema": {"type": "object", "properties": {
         "urls": {"type": "array", "items": {"type": "string"}, "description": "Page URLs to read (1-5)."},
         "objective": {"type": "string", "description": "Optional: what to look for on the pages."},
         "full_content": {"type": "boolean", "description": "Return the whole page instead of the relevant parts."}},
         "required": ["urls"]}},
]
if BACKEND == "parallel" and PARALLEL_KEY:
    TOOLS.append({"name": "research",
                  "description": "Deep research on one question; returns a cited answer (slower, uses your Parallel credits).",
                  "inputSchema": {"type": "object", "properties": {
                      "question": {"type": "string", "description": "The research question."}},
                      "required": ["question"]}})


def handle(msg):
    method, mid = msg.get("method"), msg.get("id")
    if mid is None:
        return None                                          # notification
    if method == "initialize":
        pv = (msg.get("params") or {}).get("protocolVersion") or "2025-06-18"
        return {"protocolVersion": pv, "capabilities": {"tools": {"listChanged": False}},
                "serverInfo": {"name": "qwen38-web", "version": "6.0"}}
    if method == "ping":
        return {}
    if method == "tools/list":
        return {"tools": TOOLS}
    if method == "tools/call":
        p = msg.get("params") or {}
        name, args = p.get("name"), p.get("arguments") or {}
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                args = {"query": args}
        try:
            if name in ("search", "web_search"):
                text = run_with_fallback(SEARCH, args)
            elif name in ("fetch", "web_fetch"):
                if not _urls(args):
                    raise RuntimeError("web_fetch needs at least one URL in 'urls'")
                text = run_with_fallback(FETCH, args)
            elif name in ("research", "web_research"):
                text = parallel_research(args)
            else:
                raise RuntimeError(f"unknown tool {name}")
            if len(text) > MAX_CHARS:
                text = text[:MAX_CHARS] + "\n\n[...truncated]"
            return {"content": [{"type": "text", "text": text}], "isError": False}
        except Exception as e:
            return {"content": [{"type": "text", "text": f"Web tool error: {e}"}], "isError": True}
    raise KeyError(method)


def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except ValueError:
            continue
        msgs = msg if isinstance(msg, list) else [msg]
        for m in msgs:
            if not isinstance(m, dict):
                continue
            try:
                res = handle(m)
                if res is None:
                    continue
                out = {"jsonrpc": "2.0", "id": m.get("id"), "result": res}
            except KeyError:
                out = {"jsonrpc": "2.0", "id": m.get("id"),
                       "error": {"code": -32601, "message": f"Method not found: {m.get('method')}"}}
            except Exception as e:
                out = {"jsonrpc": "2.0", "id": m.get("id"), "error": {"code": -32603, "message": str(e)}}
            sys.stdout.write(json.dumps(out, ensure_ascii=False) + "\n")
            sys.stdout.flush()


if __name__ == "__main__":
    main()
'''

if WEB_BACKEND != "off":
    open(WEB_SCRIPT, "w").write(WEB_MCP_SOURCE)
    web_env = {"WEB_BACKEND": WEB_BACKEND, "WEB_FALLBACK": "1" if AUTO_FALLBACK else "0",
               "PARALLEL_API_KEY": PKEY, "EXA_API_KEY": EKEY, "PYTHONIOENCODING": "utf-8"}
    json.dump({"mcpServers": {"web": {"command": sys.executable, "args": ["-u", WEB_SCRIPT],
                                      "env": web_env, "timeout_ms": 300000}}}, open(MCP_CONFIG, "w"))
    print(f"⏳ Testing web search ({WEB_BACKEND})... ", end="", flush=True)
    try:
        probe = "\n".join(json.dumps(m) for m in [
            {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "web_search", "arguments": {"query": "Qwen3.8 27B Hugging Face"}}}]) + "\n"
        out = subprocess.run([sys.executable, "-u", WEB_SCRIPT], input=probe, capture_output=True, text=True,
                             timeout=120, env=dict(os.environ, **web_env)).stdout
        res = [json.loads(l) for l in out.splitlines() if l.strip().startswith("{")]
        r2 = next((r for r in res if r.get("id") == 2), {}).get("result", {})
        if r2 and not r2.get("isError"):
            txt = r2["content"][0]["text"]
            print("✅" + (f"  (note: {txt.splitlines()[0][1:-1]})" if txt.startswith("[results from") else ""))
        else:
            msg = (r2.get("content") or [{"text": "no reply"}])[0]["text"]
            print(f"⚠️ {msg[:300]}\n   The tools stay enabled and retry on every call.")
    except Exception as e:
        print(f"⚠️ test failed ({e}); the tools stay enabled and retry on every call.")

# ---------------- model server ----------------
today = datetime.date.today().strftime("%A, %B %d, %Y")
system_msg = f"You are Qwen3.8-27B (HauhauCS Aggressive, uncensored), a helpful assistant. Today's date is {today}."
if WEB_BACKEND != "off":
    system_msg += (" You have web tools: use web_search for anything recent, factual or uncertain, then web_fetch "
                   "to read the most relevant pages, and cite the URLs you used.")
if MMPROJ_PATH:
    system_msg += " You can also see images that the user attaches."
UI_CONFIG = json.dumps({"systemMessage": system_msg})

def pkill(name):
    subprocess.run(["pkill", "-x", name], stderr=subprocess.DEVNULL)

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()

def spec_label(spec):
    return {"fastmtp": "FastMTP", "embedded": "built-in MTP", "off": "no MTP"}.get(spec, spec)

def start_server(ctx, kv, spec):
    cmd = [BIN, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(PORT),
           "--alias", f"Qwen3.8-27B-Aggressive-{QUANT}",
           "-ngl", "99", "--fit", "off", "-c", str(ctx), "--parallel", "1",
           "--batch-size", "2048", "--ubatch-size", "512",
           "-fa", "on", "-ctk", kv, "-ctv", kv, "--cache-ram", str(CACHE_RAM_MIB),
           "--jinja", "--ui-config", UI_CONFIG,
           "--repeat-penalty", "1.0",
           "--min-p", "0.0"]
    # Model card sampling: thinking = temp 1.0 / top_p 0.95 / top_k 20 / presence 0
    #                      non-thinking = temp 0.7 / top_p 0.80 / top_k 20 / presence 1.5
    if THINKING:
        temp, top_p, top_k, pres = 1.0, 0.95, 20, PRESENCE_PENALTY_THINKING
    else:
        temp, top_p, top_k, pres = 0.7, 0.80, 20, PRESENCE_PENALTY_NONTHINKING
    cmd += ["--temp", str(temp), "--top-p", str(top_p), "--top-k", str(top_k)]
    if pres > 0:
        cmd += ["--presence-penalty", str(pres)]
    if THINKING:
        cmd += ["--reasoning", "on", "--reasoning-effort", EFFORT,
                "--reasoning-budget", str(REASONING_BUDGET), "--reasoning-format", "deepseek"]
        if REASONING_BUDGET > 0:
            cmd += ["--reasoning-budget-message",
                    " Okay, I have thought about this enough. Time to give the final answer.\n"]
    else:
        # --reasoning off sets enable_thinking=false in the chat template (the model card's recommended way).
        cmd += ["--reasoning", "off", "--reasoning-format", "deepseek"]
    if spec == "fastmtp":
        cmd += ["--spec-type", "draft-mtp",
                "--spec-draft-model", FASTMTP_PATH,
                "--spec-draft-ngl", "all",
                "--spec-draft-n-max", str(MTP_DEPTH_FASTMTP),
                "--spec-draft-p-min", "0"]
    elif spec == "embedded":
        cmd += ["--spec-type", "draft-mtp",
                "--spec-draft-n-max", str(MTP_DEPTH_EMBEDDED),
                "--spec-draft-p-min", "0"]
    if MMPROJ_PATH and os.path.exists(MMPROJ_PATH):
        cmd += ["--mmproj", MMPROJ_PATH]
    if WEB_BACKEND != "off":
        cmd += ["--mcp-servers-config", MCP_CONFIG]
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait_ready(proc, timeout=600):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            return False
        try:
            if LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                time.sleep(1)
                return proc.poll() is None
        except requests.RequestException:
            pass
        time.sleep(2)
    return False

def log_text():
    try:
        return open(LOG, errors="replace").read()
    except OSError:
        return ""

def fail_reason():
    log = log_text().lower()
    if ("no usable gpu found" in log or "ggml_cuda_init: failed" in log
            or "no cuda-capable device" in log or "compiled without cuda" in log
            or "built without cuda" in log):
        return "no gpu"
    if any(s in log for s in ("out of memory", "cudamalloc failed", "failed to allocate", "unable to allocate")):
        return "out of memory"
    if ("expected 5120, 248320" in log
            or ("draft" in log and ("failed to load" in log or "error loading" in log))):
        return "fastmtp"
    if ("failed to create mtp context" in log or "failed to init mtp" in log
            or ("mtp" in log and "unknown argument" in log)):
        return "mtp"
    if any(("flash" in line and any(k in line for k in ("quantiz", "requires", "not supported", "cache")))
           for line in log.splitlines()):
        return "flash-attention / KV-quantization problem"
    if "couldn't bind" in log or "address already in use" in log:
        return "port busy"
    if "error while handling argument" in log or "invalid argument" in log or "unknown argument" in log:
        lines = [l for l in log.strip().splitlines() if l.strip()]
        return "bad argument: " + (lines[-1][:240] if lines else "")
    if "error loading model" in log or "failed to load model" in log:
        return "model load"
    return "see log"

def fully_on_gpu():
    log = log_text().lower()
    if ("no usable gpu found" in log or "ggml_cuda_init: failed" in log
            or "compiled without cuda" in log or "built without cuda" in log):
        return False
    m = re.search(r"offloaded (\d+)/(\d+) layers to gpu", log)
    return m is None or m.group(1) == m.group(2)

def gpu_mem():
    try:
        used, total = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                                      "--format=csv,noheader,nounits"], capture_output=True, text=True
                                     ).stdout.strip().splitlines()[0].split(",")
        return int(used), int(total)
    except Exception:
        return None, None

def mem_label(kv):
    # the cache that stores the conversation; this is NOT the model file
    return "8-bit chat memory" if kv == "q8_0" else "16-bit chat memory"

def estimate_start_ctx(kv):
    """Skip context sizes that clearly cannot fit, so a 27B model is not reloaded several times."""
    used, total = gpu_mem()
    if used is None:
        return TARGET_CONTEXT
    overhead = 650
    if MMPROJ_PATH:
        overhead += 1024
    if MTP_MODE == "fastmtp":
        overhead += 1536          # 903 MB sidecar + its own KV/context
    elif MTP_MODE == "embedded":
        overhead += 512           # extra MTP context on the target weights
    avail = (total - used) - (os.path.getsize(MODEL_PATH) / 2**20) - overhead - MIN_FREE_MIB
    per_k = 34 if kv == "q8_0" else 68
    max_ctx = int(max(0, avail) / per_k * 1024)
    fitting = [c for c in CTX_STEPS if c <= max_ctx]
    start = max(fitting) if fitting else CTX_STEPS[-1]
    return min(start, TARGET_CONTEXT)

def attempts_from(start_ctx):
    sizes = [start_ctx] + [c for c in CTX_STEPS if c < start_ctx]
    att = [(c, KV_CACHE_TYPE) for c in sizes]
    if KV_CACHE_TYPE != "f16":
        att += [(c, "f16") for c in (16384, 8192, 4096) if c <= start_ctx]   # last-resort fallbacks
    seen, out = set(), []
    for x in att:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def launch_with_best_context(target_ctx):
    global PORT
    spec = MTP_MODE
    port_retries = 0
    start_ctx = estimate_start_ctx(KV_CACHE_TYPE)
    if start_ctx < target_ctx:
        print(f"   ℹ️ Free-VRAM estimate: {start_ctx:,} tokens fits (you picked {target_ctx:,}).")
    att = attempts_from(start_ctx)
    i = 0
    while i < len(att):
        ctx, kv = att[i]
        print(f"   trying {ctx:>7,} tokens ({mem_label(kv)}, {spec_label(spec)}) ... ", end="", flush=True)
        proc = start_server(ctx, kv, spec)
        ok, reason = wait_ready(proc), ""
        if ok:
            early = fail_reason()
            if early == "no gpu":
                ok, reason = False, early
            elif not fully_on_gpu():
                ok, reason = False, "model did not fully fit on the GPU"
            else:
                used, total = gpu_mem()
                if used is not None and total - used < MIN_FREE_MIB:
                    ok, reason = False, f"fits, but only {total-used} MiB GPU memory left (unsafe for long chats)"
        else:
            reason = fail_reason()
        if ok:
            print("✅")
            return proc, ctx, kv, spec
        print(f"❌ ({reason})")

        # A failed accelerator is unrelated to context size: drop it and retry the same size.
        if reason == "fastmtp" and spec == "fastmtp":
            stop(proc)
            spec = "embedded"
            print("   ↳ FastMTP could not load — retrying with built-in MTP.")
            continue
        if reason == "mtp" and spec in ("fastmtp", "embedded"):
            stop(proc)
            spec = "off"
            print("   ↳ MTP could not start — retrying without speculative decoding.")
            continue

        # Quantized KV problem: retry the same context with a 16-bit cache, then smaller sizes.
        if reason.startswith("flash") and kv != "f16":
            stop(proc)
            sizes = [c for c, _ in att[i:] if c <= ctx]
            att = att[:i] + [(c, "f16") for c in sizes]
            print("   ↳ retrying the same and smaller contexts with 16-bit chat memory.")
            continue

        if reason == "port busy":
            stop(proc)
            PORT = free_port(PORT + 1)
            port_retries += 1
            if port_retries > 3:
                break
            print(f"   ↳ port was busy, switching to {PORT} and retrying.")
            continue

        if reason.startswith("bad argument") or reason in ("model load", "no gpu"):
            stop(proc)
            break
        stop(proc)
        i += 1
        time.sleep(2)
    print(log_text()[-3000:])
    raise RuntimeError("Could not start the model — see the log above. Try a smaller quant in Cell 1 "
                       "(IQ3_XS or Q2_K_P on a T4), turn VISION/FastMTP off, or pick a smaller context.")

# ---------------- public link ----------------
def tunnel_ready():
    try:
        return LOCAL.get(f"http://127.0.0.1:{METRICS_PORT}/ready", timeout=3).status_code == 200
    except requests.RequestException:
        return False

def url_answers(url):
    try:
        r = requests.get(url + "/health", timeout=8, headers={"ngrok-skip-browser-warning": "1"})
        return r.status_code == 200
    except requests.RequestException:
        return False

def start_cloudflare():
    pkill("cloudflared")
    time.sleep(1)
    for attempt in range(3):
        p = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                              "--protocol", "http2", "--no-autoupdate",
                              "--metrics", f"127.0.0.1:{METRICS_PORT}"],
                             stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(60):
            if p.poll() is not None:
                break
            try:
                found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
                         if not u.startswith("https://api.")]
            except OSError:
                found = []
            if found:
                url = found[0]
                break
            time.sleep(1)
        if url:
            for _ in range(30):   # wait until the link actually answers
                if p.poll() is not None:
                    break
                if tunnel_ready() and url_answers(url):
                    return p, url
                time.sleep(2)
            if p.poll() is None and tunnel_ready():
                return p, url          # connected; the edge may just need a few more seconds
        stop(p)
        time.sleep(3)
    return None, None

def start_ngrok():
    try:
        from pyngrok import ngrok, conf
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=False)
        try:
            from pyngrok import ngrok, conf
        except ImportError:
            print("   ❌ pyngrok could not be installed — using Cloudflare instead.")
            return None, None
    import io, logging, contextlib
    logging.getLogger("pyngrok").setLevel(logging.CRITICAL)   # keep the cell output clean
    for attempt in range(3):
        try:
            try:
                ngrok.kill()
            except Exception:
                pass
            conf.get_default().auth_token = NGROK_TOKEN
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                t = ngrok.connect(f"127.0.0.1:{PORT}", "http")
            url = t.public_url.replace("http://", "https://")
            for _ in range(15):
                if url_answers(url):
                    break
                time.sleep(2)
            return t, url
        except Exception as e:
            msg = str(e)
            if "authtoken" in msg.lower() or "ERR_NGROK_105" in msg or "ERR_NGROK_107" in msg:
                print("   ❌ ngrok rejected the token — copy it again from dashboard.ngrok.com → Your Authtoken.")
                break          # wrong token: retrying won't help
            print(f"   ngrok attempt {attempt + 1} failed: {msg.strip()[:200]}")
            time.sleep(5)
    return None, None

def link_alive(handle):
    if handle is None:
        return False
    if LINK_MODE == "ngrok":
        try:
            from pyngrok import ngrok
            return ngrok.get_ngrok_process().proc.poll() is None and len(ngrok.get_tunnels()) > 0
        except Exception:
            return False
    return handle.poll() is None and tunnel_ready()

def start_link():
    global LINK_MODE
    try:
        if LINK_MODE == "ngrok":
            h, u = start_ngrok()
            if u:
                return h, u
            print("⚠️ ngrok didn't start — falling back to a Cloudflare link.")
            LINK_MODE = "cloudflare"
        return start_cloudflare()
    except Exception as e:
        print(f"⚠️ Could not start the public link: {e}")
        return None, None

def stop_link(handle):
    if handle is None:
        return
    if LINK_MODE == "ngrok":
        try:
            from pyngrok import ngrok
            ngrok.kill()
        except Exception:
            pass
    else:
        stop(handle)

def show_link(url, title):
    display(HTML(f'<div style="font-size:16px;margin:8px 0">{title} <b><a href="{url}" target="_blank">{url}</a></b></div>'))

# ================= main =================
pkill("llama-server")
pkill("cloudflared")
# On a re-run, wait for the previous 27B server to actually release its VRAM before estimating.
for _ in range(30):
    used, _total = gpu_mem()
    if used is None or used < 2048:
        break
    time.sleep(1)
else:
    print("⚠️ Another process is still using GPU memory; the context estimate will be conservative.")
time.sleep(1)
PORT, METRICS_PORT = free_port(8090), free_port(20241)

print(f"⏳ Loading Qwen3.8-27B {QUANT} with up to {TARGET_CONTEXT:,} tokens of context...")
server, CTX, KV, SPEC = launch_with_best_context(TARGET_CONTEXT)

used, total = gpu_mem()
if not THINKING:
    think_txt = "OFF (non-thinking sampling: temp 0.7, top_p 0.80, presence 1.5)"
elif REASONING_BUDGET < 0:
    think_txt = "Max (xhigh, unlimited)"
else:
    think_txt = f"{REASONING.split()[0]} ({EFFORT}, up to {REASONING_BUDGET:,} tokens)"
print(f"\n✅ Model ready  |  model: {QUANT}  |  context: {CTX:,} tokens ({mem_label(KV)})"
      f"  |  reasoning: {think_txt}  |  MTP: {spec_label(SPEC)}")
if CTX < TARGET_CONTEXT:
    print(f"   ℹ️ {TARGET_CONTEXT:,} didn't fit in GPU memory with {QUANT}; using {CTX:,}. "
          f"For more context pick a smaller quant in Cell 1 (IQ3_XS/Q2_K_P on a T4).")
if SPEC != MTP_MODE:
    print(f"   ℹ️ MTP fallback: {spec_label(MTP_MODE)} → {spec_label(SPEC)}.")
print(f"   web search: {('ON · ' + WEB_BACKEND + (' + auto-fallback' if AUTO_FALLBACK else '')) if WEB_BACKEND != 'off' else 'OFF'}"
      f"  |  vision: {'ON' if MMPROJ_PATH else 'off'}"
      f"  |  GPU memory: {used if used is not None else '?'} / {total if total is not None else '?'} MiB\n")

tunnel, public_url = None, None
try:
    from google.colab import output
    # Colab's private new-tab links (*.prod.colab.dev) now return 404 in modern browsers,
    # so only the in-notebook view is shown; use the public link for a full tab.
    if SHOW_INLINE:
        print("💬 Chat inside the notebook (private, only you):")
        output.serve_kernel_port_as_iframe(PORT, height=850)
    if not SHOW_INLINE and LINK_MODE == "off":
        print("⚠️ Both the inline chat and the public link are off — there is no chat UI. Set SHOW_INLINE=True or pick a PUBLIC_LINK.")
    if LINK_MODE != "off":
        print(f"⏳ Opening the public link ({LINK_MODE})...")
        tunnel, public_url = start_link()
        if public_url:
            show_link(public_url, "🌍 Public link:")
            if LINK_MODE == "ngrok":
                print("   (ngrok shows a one-time “Visit Site” page — click it. This link stays the same next time.)")
            else:
                print("   (If it says it can't be reached, wait 30 s and refresh, or open it in an incognito window.)")
        else:
            print("⚠️ The public link didn't start yet — it keeps retrying below. The chat above works meanwhile.")

    print("\nHow to use: open a link → type your message. Long text/code (1,000+ lines) is fine: paste it or attach the file.")
    if WEB_BACKEND != "off":
        print("Web search: the first time the model searches, the chat asks permission — click “Always allow”.")
    print("Reasoning can also be toggled per chat in the UI; the sampling above is set for your Cell 2 choice.")
    print("Keep this cell running. Press ■ (stop) to shut everything down.\n")

    start, last_report = time.time(), 0
    server_restarts = tunnel_restarts = bad_ticks = 0
    while True:
        time.sleep(20)

        # 1) model server (auto-restart; if it ran out of memory, restart with the next smaller context)
        if server.poll() is not None:
            if server_restarts >= MAX_SERVER_RESTARTS:
                print("❌ Model server keeps crashing. Last log lines:\n")
                print(log_text()[-3000:])
                break
            server_restarts += 1
            reason = fail_reason()
            if reason == "port busy":
                print("⚠️ The server port is busy; retrying on the same port to keep the chat and public link working.")
            if reason == "out of memory":
                smaller = [c for c in CTX_STEPS if c < CTX]
                if smaller:
                    print(f"⚠️ Ran out of GPU memory at {CTX:,} tokens — restarting with {smaller[0]:,}.")
                    CTX = smaller[0]
                else:
                    print("❌ Out of GPU memory even at the smallest context. Last log lines:\n")
                    print(log_text()[-3000:])
                    break
            elif reason == "fastmtp" and SPEC == "fastmtp":
                SPEC = "embedded"
                print("⚠️ FastMTP stopped — restarting with built-in MTP.")
            elif reason == "mtp" and SPEC in ("fastmtp", "embedded"):
                SPEC = "off"
                print("⚠️ MTP stopped — restarting without speculative decoding.")
            elif reason == "model load":
                print("❌ The model file could not be loaded. Run Cell 1 again to re-download it.\n")
                print(log_text()[-3000:])
                break
            elif reason == "no gpu":
                print("❌ CUDA is not available to llama-server. Re-run Cell 1 (it needs a GPU runtime).\n")
                print(log_text()[-3000:])
                break
            elif reason.startswith("bad argument"):
                print("❌ The server rejected an argument:\n" + reason)
                print(log_text()[-3000:])
                break
            print(f"⚠️ Model server stopped — restarting ({server_restarts}/{MAX_SERVER_RESTARTS})...")
            server = start_server(CTX, KV, SPEC)
            if wait_ready(server):
                if fail_reason() == "no gpu" or not fully_on_gpu():
                    print("⚠️ Restart did not use the GPU; stopping it and retrying.")
                    stop(server)
                else:
                    print("✅ Model server is back. Your chats are kept in your browser; just resend the last message.")
            else:
                print(f"⚠️ Restart did not become ready ({fail_reason()}); stopping it and retrying.")
                stop(server)
            continue

        # 2) public link
        if LINK_MODE != "off":
            bad_ticks = 0 if link_alive(tunnel) else bad_ticks + 1
            if tunnel is None or bad_ticks >= 3:
                tunnel_restarts += 1
                print(f"⚠️ Public link dropped — reconnecting (#{tunnel_restarts})...")
                stop_link(tunnel)
                tunnel, new_url = start_link()
                bad_ticks = 0
                if new_url and new_url != public_url:
                    public_url = new_url
                    show_link(public_url, "🌍 NEW public link (the old one no longer works):")
                elif new_url:
                    print("✅ Public link reconnected (same address).")

        # 3) heartbeat every 10 min
        mins = int((time.time() - start) // 60)
        if mins - last_report >= 10:
            last_report = mins
            print(f"⏱ Up {mins} min | context: {CTX:,} | MTP: {spec_label(SPEC)}"
                  f" | server restarts: {server_restarts} | link restarts: {tunnel_restarts}")
except KeyboardInterrupt:
    print("Stopping...")
finally:
    stop_link(tunnel)
    stop(server)
    print("Shut down.")


## Troubleshooting

- **"No GPU found"** → Runtime → Change runtime type → T4 GPU → Save, then run Cell 1 again.
- **Q4_K_P won't fit on a T4** → expected. Cell 1 auto-falls back *before* downloading. For more context pick **IQ3_XS** or **Q2_K_P**, or leave `AUTO_QUANT_FALLBACK` ticked.
- **Context is lower than you picked** → that's the most this GPU can hold with this quant and MTP setting. Pick a smaller quant, turn VISION/FastMTP off, set MTP → Off, or start with 65536/32768.
- **Out of memory while chatting** → Cell 2 restarts with the next smaller context automatically. To avoid it, start lower.
- **"Unexpected reasoning effort high/max"** → the model's chat template accepts only **xhigh / medium / low**. Use the notebook picker; if you call the API yourself, send `xhigh` for Max/High, `medium`, or `low`.
- **FastMTP build failed** → Cell 1 falls back to the stock build and built-in MTP. FastMTP is optional; check `/content/build.log`.
- **"expected 5120, 248320, got 5120, 32768"** → the binary is unpatched but a FastMTP sidecar was supplied. Re-run Cell 1 with FastMTP ticked (patched build) or turn FastMTP off.
- **MTP failed to load** → Cell 2 automatically retries with built-in MTP, then without MTP. The status line shows which one is active.
- **Web search says rate-limited** → it falls back automatically. For heavy use, add a free key from platform.parallel.ai in `PARALLEL_API_KEY` (or a Colab secret with that name).
- **The model rambles or loops while thinking** → pick Medium or Low, or set `PRESENCE_PENALTY_THINKING = 1.0` in Cell 2.
- **Public link doesn't open** → wait ~30 s and refresh, try an incognito window, or switch to **ngrok** (free token) for a stable link. The chat shown inside the notebook always works.
- **"q8_0 / 8-bit chat memory"** → that's the conversation cache, not the model. Your chosen quant is shown as `model:`.
- **Download or build failed** → just run Cell 1 again; finished parts are reused. Logs: `/content/build.log` and `/content/llama-server.log`.
- **Session disconnected** → reconnect, run Cell 1 then Cell 2 (tick `USE_DRIVE_CACHE` to skip the rebuild next time). You'll get a new public link.
- **Slow generation on a T4** → expected for a 27B model. Use IQ3_XS/Q2_K_P, keep MTP on, and use a smaller context. An L4/A100 is much faster.
- **Vision runs out of memory** → untick VISION (it adds ~1 GB) or pick a smaller quant.
